[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/15_mlp_solution.ipynb)

# ✅ Solution: mlp

Implement the **SwiGLU MLP** (feed-forward network) used in modern LLMs like LLaMA.

$$\text{SwiGLU}(x) = \text{down\_proj}\big(\text{SiLU}(\text{gate\_proj}(x)) \odot \text{up\_proj}(x)\big)$$

where $\text{SiLU}(x) = x \cdot \sigma(x)$

### Signature
```python
class SwiGLUMLP(nnx.Module):
    def __init__(self, d_model: int, d_ff: int): ...
    def forward(self, x: jax.Array) -> jax.Array: ...
```

### Requirements
- Inherit from `nnx.Module`
- `self.gate_proj`: `nn.Linear(d_model, d_ff)`
- `self.up_proj`: `nn.Linear(d_model, d_ff)`
- `self.down_proj`: `nn.Linear(d_ff, d_model)`
- Activation: **SiLU** (a.k.a. Swish) — `F.silu` or implement as `x * jnp.sigmoid(x)`

### Why SwiGLU?
Unlike the classic `Linear → ReLU/GELU → Linear` FFN, SwiGLU uses a **gating mechanism**:
the gate projection controls information flow, while the up projection provides the content.
This consistently outperforms standard FFNs in practice (PaLM, LLaMA, Mistral all use it).


In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge')
except ImportError:
    pass


In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx
import math


In [ ]:
# ✅ SOLUTION

import jax
from flax import nnx
class SwiGLUMLP(nnx.Module):
    def __init__(self, d_model, d_ff, *, rngs):
        self.gate_proj = nnx.Linear(d_model, d_ff, rngs=rngs)
        self.up_proj = nnx.Linear(d_model, d_ff, rngs=rngs)
        self.down_proj = nnx.Linear(d_ff, d_model, rngs=rngs)
    def __call__(self, x_BLD):
        # x_BLD -> gate/up_BLF -> out_BLD
        return self.down_proj(jax.nn.silu(self.gate_proj(x_BLD)) * self.up_proj(x_BLD))


In [ ]:
# Verify
print(SwiGLUMLP)


In [ ]:
from jax_judge import check
check("mlp")
